# SaaS Cohort Retention Analysis
## Milestone 3: Cohort Retention (Python + Heatmap)

**Author:** Anuj Saini  
**Date:** July 7, 2026  

This notebook tracks the retention of monthly paid subscriber cohorts over time, visualizes retention using a seaborn heatmap, and highlights the core operational retention story of the business.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import psycopg2
from dotenv import load_dotenv
import os

# Configure plotting style
sns.set_theme(style="white")
plt.rcParams["figure.figsize"] = (16, 10)

# Load database env
load_dotenv(dotenv_path="../.env")

### 1. Load Data

We fetch companies and subscriptions from the analytical database.

In [ ]:
host = os.getenv("host")
port = os.getenv("port")
database = os.getenv("database")
username = os.getenv("username")
password = os.getenv("password")

conn = psycopg2.connect(
    host=host,
    port=port,
    database=database,
    user=username,
    password=password
)

df_companies = pd.read_sql("SELECT id, signup_date FROM saas.legacy_companies;", conn)
df_subs = pd.read_sql("SELECT id, company_id, plan, start_date, end_date, mrr, status FROM saas.legacy_subscriptions;", conn)
conn.close()

df_companies['signup_date'] = pd.to_datetime(df_companies['signup_date'])
df_subs['start_date'] = pd.to_datetime(df_subs['start_date'])
df_subs['end_date'] = pd.to_datetime(df_subs['end_date'])

print(f"Loaded {df_companies.shape[0]} companies and {df_subs.shape[0]} subscriptions.")

### 2. Compute Cohort Retention Table

We define the cohort month based on `signup_date` and check if the company was active on a paid, non-trial plan during each target month offset.

In [ ]:
# Cohort group
df_companies['cohort'] = df_companies['signup_date'].dt.to_period('M')

records = []
for idx, row in df_companies.iterrows():
    comp_id = row['id']
    cohort_start = row['cohort']
    c_subs = df_subs[df_subs['company_id'] == comp_id]
    
    for offset in range(19):
        target_period = cohort_start + offset
        if target_period > pd.Period('2026-04', freq='M'):
            continue
        
        m_start = target_period.to_timestamp(how='start')
        m_end = target_period.to_timestamp(how='end')
        
        # Active paid during month C+offset
        active = c_subs[
            (c_subs['plan'] != 'free') &
            (c_subs['status'] != 'trial') &
            (c_subs['start_date'] <= m_end) &
            ((c_subs['end_date'].isnull()) | (c_subs['end_date'] >= m_start))
        ]
        is_active = len(active) > 0
        records.append({
            'company_id': comp_id,
            'cohort': str(cohort_start),
            'offset': offset,
            'is_active': int(is_active)
        })

df_cohort = pd.DataFrame(records)

# Include only companies that started as paid subscribers in Month 0
companies_month0 = df_cohort[(df_cohort['offset'] == 0) & (df_cohort['is_active'] == 1)]['company_id'].unique()
df_cohort_filtered = df_cohort[df_cohort['company_id'].isin(companies_month0)]

cohort_sizes = df_cohort_filtered[df_cohort_filtered['offset'] == 0].groupby('cohort')['company_id'].count()
cohort_active = df_cohort_filtered.groupby(['cohort', 'offset'])['is_active'].sum().unstack()
cohort_retention = cohort_active.div(cohort_sizes, axis=0) * 100

print("Cohort size and active company counts pivot table:")
cohort_active

### 3. Render Seaborn Heatmap

We plot the retention percentages using a single-hue yellow-green (`YlGn`) color palette, leaving cells in the future empty (NaN masked).

In [ ]:
# Update index labels to include cohort sizes
cohort_labels = [f"{idx} (n={cohort_sizes[idx]})" for idx in cohort_retention.index]
cohort_retention.index = cohort_labels

plt.figure(figsize=(16, 10))
mask = cohort_retention.isnull()

sns.heatmap(
    cohort_retention, 
    annot=True, 
    fmt=".1f", 
    cmap="YlGn", 
    mask=mask, 
    cbar_kws={'label': 'Retention Rate (%)'},
    linewidths=0.5,
    annot_kws={'size': 10}
)

plt.title('Paid Subscription Cohort Retention Heatmap', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Months Since Signup (Offset)', fontsize=12, labelpad=10)
plt.ylabel('Cohort Signup Month (Size)', fontsize=12, labelpad=10)
plt.tight_layout()
plt.show()